# 05 — Classical computer vision + machine learning

Runs E1–E5 on the **existing** 3-seed × 4-fold split manifests. The pipeline crops each image with the shared `watermark_safe_crop`, selects hyperparameters on validation data, fits on training data, and evaluates each test partition once. It writes predictions and recomputable metrics to `results/classical_ml/`.

In Colab, clone or mount this repository, then set `PROJECT_ROOT` below. Make the raw image folders available at `data/raw/{FN,-N,-P,-K}` (or change `IMAGE_ROOT`). The raw images are not included in the repository. No split regeneration or fabricated results are performed.

In [ ]:
from pathlib import Path
import sys

# In Colab, change this to the cloned/mounted repository directory.
PROJECT_ROOT = Path.cwd()
IMAGE_ROOT = PROJECT_ROOT / "data" / "raw"
SPLITS_DIR = PROJECT_ROOT / "data" / "splits"
OUTPUT_DIR = PROJECT_ROOT / "results" / "classical_ml"

if not (PROJECT_ROOT / "lettuce_benchmark").is_dir():
    raise FileNotFoundError("Set PROJECT_ROOT to the repository root")
if not IMAGE_ROOT.is_dir():
    raise FileNotFoundError(f"Place the raw class image folders at {IMAGE_ROOT}")
if not SPLITS_DIR.is_dir():
    raise FileNotFoundError(f"Shared split manifests not found at {SPLITS_DIR}")
sys.path.insert(0, str(PROJECT_ROOT))
print(f"Repository: {PROJECT_ROOT}")
print(f"Images: {IMAGE_ROOT}")
print(f"Shared manifests: {SPLITS_DIR}")

Install the project dependencies if the current runtime does not already have them. From a terminal, the repository's `requirements.txt` is the canonical dependency list; for a lightweight Colab setup, the following covers this pipeline:

In [ ]:
%pip install -q numpy pandas scikit-learn opencv-python-headless scikit-image pillow

## Inspect the shared manifests

This checks their existing contents and does not create or modify any split.

In [ ]:
import pandas as pd

manifest_paths = sorted(SPLITS_DIR.glob("seed_*/fold_*.csv"))
assert len(manifest_paths) == 12, f"Expected 12 shared manifests; found {len(manifest_paths)}"
manifest_summary = []
for path in manifest_paths:
    frame = pd.read_csv(path)
    manifest_summary.append({
        "manifest": str(path.relative_to(PROJECT_ROOT)),
        "images": len(frame),
        "train": int((frame["partition"] == "train").sum()),
        "validation": int((frame["partition"] == "validation").sum()),
        "test": int((frame["partition"] == "test").sum()),
    })
pd.DataFrame(manifest_summary)

## Preview feature extraction

Features are computed only after the shared watermark-safe crop. The preview reports dimensions and finite-value checks without training a model.

In [ ]:
import numpy as np
from PIL import Image
from models.classical_svm.feature_extraction import extract_feature_groups

preview_manifest = pd.read_csv(manifest_paths[0])
preview_row = preview_manifest.iloc[0]
preview_path = IMAGE_ROOT / preview_row["image_path"]
with Image.open(preview_path) as image:
    feature_groups = extract_feature_groups(image)
feature_summary = {
    name: {"dimensions": values.size, "all_finite": bool(np.isfinite(values).all())}
    for name, values in feature_groups.items()
}
feature_summary

## Run all five experiments

E1–E4 use balanced RBF SVMs; E5 uses a Random Forest. StandardScaler is included in each training pipeline. The selected hyperparameters are chosen by validation macro-F1. Feature-extraction timing is recorded separately from classifier prediction timing.

In [ ]:
from models.classical_svm.pipeline import run_experiments

results = run_experiments(
    image_root=IMAGE_ROOT,
    splits_dir=SPLITS_DIR,
    output_dir=OUTPUT_DIR,
)
print(f"Saved experiment outputs under {OUTPUT_DIR}")
sorted(results)

## Inspect saved outputs

`metrics.csv` contains fold-level test results, per-seed out-of-fold results, and mean/sample-standard-deviation rows across seeds, including per-class precision, recall, and F1. Confusion matrices and per-run selected parameters are saved alongside the combined predictions.

In [ ]:
experiment = "E1"
metrics = pd.read_csv(OUTPUT_DIR / "metrics.csv")
predictions = pd.read_csv(OUTPUT_DIR / "predictions.csv")
efficiency = pd.read_csv(OUTPUT_DIR / "efficiency.csv")
print("OOF mean and sample standard deviation:")
display(metrics.loc[
    (metrics["experiment"] == experiment)
    & metrics["level"].isin(["seed_mean", "seed_std"])
])
print("Prediction schema and count:", list(predictions.columns), len(predictions))
efficiency.loc[efficiency["experiment"] == experiment].head()